# Setup check — run this at home, after the download notebook

Thirty seconds. It tells you which modules will run on your machine, so that any problem is found now rather than in the first ten minutes of the session.


In [ ]:
import importlib

print('Core packages — every module needs these:')
missing = []
for name, why in [('numpy', 'numbers'), ('pandas', 'tables'), ('matplotlib', 'figures'),
                  ('sklearn', 'the models'), ('scipy', 'statistics')]:
    try:
        importlib.import_module(name)
        print(f'  {name:<14} OK   ({why})')
    except ImportError:
        missing.append(name)
        print(f'  {name:<14} MISSING  <- install the environment first')

print('\nOptional extras — only some cells need these:')
for name, why in [('torch', 'module A: the CNN (there is an automatic fallback without it)'),
                  ('lifelines', 'module E: proper Cox survival models (fallback available)'),
                  ('rdkit', 'module H: drawing molecules and true scaffolds'),
                  ('ipywidgets', 'checkbox selection in the download notebook'),
                  ('shap', 'nothing — we compute exact Shapley values ourselves')]:
    try:
        importlib.import_module(name)
        print(f'  {name:<14} available   ({why})')
    except ImportError:
        print(f'  {name:<14} not installed   ({why})')

if missing:
    print('\n*** Install the core packages before the session: ***')
    print('    conda env create -f environment.yml     (or)')
    print('    pip install numpy pandas matplotlib scikit-learn scipy jupyter')


In [ ]:
from pathlib import Path
import sys
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(repo_root / 'src'))
from data_registry import MODULES, SOURCES, module_ids

print('Which modules can you run right now?\n')
ready = []
for module in module_ids():
    entry = MODULES[module]
    have_raw = all((repo_root / SOURCES[key]['path']).exists() for key in entry['sources'])
    have_derived = all((repo_root / name).exists() for name in entry['derived'])
    if have_derived:
        ready.append(module)
        state = 'ready to go'
    elif have_raw:
        state = 'downloaded, will prepare itself when you open the notebook'
        ready.append(module)
    else:
        state = 'NOT ready — run 00_download_data.ipynb'
    print(f"  {module}  {entry['title']:<38} {state}")

print('\nRunnable now:', ', '.join(ready) if ready else 'none yet')
print('\nYou only need two modules. If your two are listed above, you are set.')


In [ ]:
# A 10-second end-to-end test: load a module, split, train, score, draw something.
try:
    import matplotlib.pyplot as plt
    import plots
    from data import load_data
    from models import split_data, train_model, evaluate

    candidates = [m for m in ready if m in ('C', 'E', 'H', 'D')]
    module = candidates[0] if candidates else (ready[0] if ready else None)
    if module is None:
        print('No module data yet — run 00_download_data.ipynb first.')
    else:
        frame = load_data(module)
        print(f'Loaded module {module}: {frame.shape[0]} rows x {frame.shape[1]} columns')
        numbers = frame.select_dtypes('number')
        plots.plot_missingness(frame, title=f'Module {module}: missing values')
        plt.show()
        print('\nEverything works. See you at the session.')
except Exception as error:
    print('Something is not right:', type(error).__name__, error)
    print('Send this message to the instructor before the session.')
